# Filipino Tokenizer — Demo

A quick walkthrough of the `TagalogTokenizer`: morphology-aware BPE for Filipino text.

**What you'll see:**
1. Train a tokenizer on a small Filipino corpus
2. Encode and decode text
3. Inspect subword tokens and morpheme segmentation
4. Save and reload the trained tokenizer
5. See how the same root is shared across inflected forms

In [10]:
import sys, os, tempfile
sys.path.insert(0, os.path.abspath(".."))

from filipino_tokenizer.tagalog import TagalogTokenizer, TagalogSegmenter

## 1. Training Corpus

A plain UTF-8 text file with one sentence per line.
In practice you would use a large corpus (e.g. WikiText-TL-39).
Here we use 15 sentences to keep the demo fast.

In [11]:
CORPUS = """\
Kumain siya ng pagkain sa hapagkainan.
Ang mga bata ay masayang naglalaro sa labas ng bahay.
Maganda ang panahon ngayon kaya lumabas kami para maglakad.
Bumili ako ng mga prutas sa palengke kahapon.
Nagluluto ang nanay ng masarap na adobo para sa buong pamilya.
Pumunta kami sa simbahan tuwing Linggo ng umaga.
Nagbabasa ang mga estudyante ng libro sa silid-aklatan.
Kumanta ang mga bata sa programa ng paaralan nila.
Umuulan kaya nagdala ako ng payong at kapote.
Naglinis kami ng bahay bago dumating ang mga bisita.
Nagtatrabaho ang tatay sa opisina araw-araw.
Kinain niya ang lahat ng pagkain sa mesa.
Pinakamahusay ang ginawa niya sa pagtatanghal.
Natutulog na ang sanggol sa kuna niya.
Gumagawa siya ng takdang-aralin bago matulog.
"""

tmpdir = tempfile.mkdtemp()
corpus_path = os.path.join(tmpdir, "corpus.txt")
with open(corpus_path, "w", encoding="utf-8") as f:
    f.write(CORPUS)

print(f"Corpus written to {corpus_path}")
print(f"Lines: {CORPUS.strip().count(chr(10)) + 1}")

Corpus written to C:\Users\JOHNPA~1\AppData\Local\Temp\tmpnou52m43\corpus.txt
Lines: 15


## 2. Train the Tokenizer

`vocab_size` controls how many BPE merge rules are learned.
On a small corpus the actual vocabulary will be smaller than the target.

In [12]:
tok = TagalogTokenizer()
tok.train(corpus_path, vocab_size=500)

print(f"Vocabulary size : {len(tok.bpe.vocab)}")
print(f"Merge rules     : {len(tok.bpe.merges)}")

Vocabulary size : 287
Merge rules     : 187


## 3. Encode and Decode

`encode()` returns a list of integer token IDs — the format expected by neural network models.
`decode()` reconstructs the original (lowercased) text.

In [13]:
sentence = "Kumain siya ng pagkain."

ids = tok.encode(sentence)
decoded = tok.decode(ids)

print(f"Input   : {sentence}")
print(f"IDs     : {ids}")
print(f"Decoded : {decoded}")
print(f"Match   : {decoded == sentence.lower()}")

Input   : Kumain siya ng pagkain.
IDs     : [79, 99, 115, 99, 133, 4, 154, 4, 100, 4, 125, 99, 145, 18]
Decoded : kumain siya ng pagkain.
Match   : True


## 4. Inspect Subword Tokens

`tokenize()` returns the string representation of each token instead of IDs.
This is useful for understanding how the tokenizer is splitting text.

In [14]:
examples = [
    "Kumain siya ng pagkain.",
    "Maganda ang panahon ngayon.",
    "Pinakamahusay ang ginawa niya.",
]

for text in examples:
    tokens = tok.tokenize(text)
    print(f"  {text}")
    print(f"  -> {tokens}")
    print()

  Kumain siya ng pagkain.
  -> ['k', '▁', 'um', '▁', 'ain', ' ', 'siya', ' ', 'ng', ' ', 'pag', '▁', 'kain', '.']

  Maganda ang panahon ngayon.
  -> ['ma', '▁', 'ganda', ' ', 'ang', ' ', 'pa', '▁', 'nahon', ' ', 'ngayon', '.']

  Pinakamahusay ang ginawa niya.
  -> ['pinaka', '▁', 'ma', '▁', 'husay', ' ', 'ang', ' ', 'gi', '▁', 'nawa', ' ', 'niya', '.']



## 5. Morphological Segmenter

The `TagalogSegmenter` can be used on its own to decompose words into morphemes.
This is the linguistic step that runs *before* BPE — it tells BPE where the boundaries are.

In [15]:
seg = TagalogSegmenter()

words = [
    "kumain",        # infix -um- + root kain
    "pagkain",       # prefix pag- + root kain
    "kinain",        # infix -in- + root kain
    "pagkainan",     # circumfix pag- -an + root kain
    "pinakamahusay", # stacked prefixes: pinaka- + ma- + root husay
    "maganda",       # prefix ma- + root ganda
    "kasiyahan",     # circumfix ka- -han + root siya
    "pangalan",      # frozen form -- NOT segmented
    "computer",      # loan word -- NOT segmented
]

print(f"  {'Word':<20} Morphemes")
print(f"  {'-'*20} ---------")
for word in words:
    morphemes = seg.segment(word)
    print(f"  {word:<20} {morphemes}")

  Word                 Morphemes
  -------------------- ---------
  kumain               ['um', 'kain']
  pagkain              ['pag', 'kain']
  kinain               ['in', 'kain']
  pagkainan            ['pag', 'kain', 'an']
  pinakamahusay        ['pinaka', 'ma', 'husay']
  maganda              ['ma', 'ganda']
  kasiyahan            ['ka', 'siya', 'han']
  pangalan             ['pangalan']
  computer             ['computer']


## 6. Root Sharing Across Inflections

Because the tokenizer respects morpheme boundaries, the root *kain* behaves
consistently across inflected forms — but *how* depends on the affix type:

- **Prefix/suffix forms** (`pagkain`): root appears intact in the surface text → same IDs every time.
- **Infix forms** (`kumain`, `kinain`): the infix is inserted *inside* the root after the first consonant,
  so the surface is `k▁um▁ain`. The root characters are split around the infix — this is correct
  Filipino morphology, not a bug. The segmenter still identifies `kain` as the root, and BPE
  never merges across the `▁` markers, but the root cannot appear as contiguous token IDs.

In [16]:
kain_forms = [
    ("kain",    "bare root"),
    ("kumain",  "infix -um-"),
    ("pagkain", "prefix pag-"),
    ("kinain",  "infix -in-"),
]
kain_ids = tok.bpe.encode("kain")

print(f"Bare 'kain' encodes to: {kain_ids}")
print()
print(f"  {'Word':<12} {'Affix type':<14} {'Morphemes':<20} {'Surface form':<18} {'Root in morphemes?':<20} {'Root IDs contiguous?'}")
print(f"  {'-'*12} {'-'*14} {'-'*20} {'-'*18} {'-'*20} {'-'*20}")

for word, affix_type in kain_forms:
    morphemes = seg.segment(word)
    annotated = tok._segment_line(word)[0]   # surface form with boundary markers
    ids = tok.encode(word)

    root_in_morphemes = "kain" in morphemes

    contiguous = any(
        ids[i:i+len(kain_ids)] == kain_ids
        for i in range(len(ids) - len(kain_ids) + 1)
    )

    print(f"  {word:<12} {affix_type:<14} {str(morphemes):<20} {annotated:<18} {'yes':<20} {'yes' if contiguous else 'no (infix splits root)'}") 

Bare 'kain' encodes to: [145]

  Word         Affix type     Morphemes            Surface form       Root in morphemes?   Root IDs contiguous?
  ------------ -------------- -------------------- ------------------ -------------------- --------------------
  kain         bare root      ['kain']             kain               yes                  yes
  kumain       infix -um-     ['um', 'kain']       k▁um▁ain           yes                  no (infix splits root)
  pagkain      prefix pag-    ['pag', 'kain']      pag▁kain           yes                  yes
  kinain       infix -in-     ['in', 'kain']       k▁in▁ain           yes                  no (infix splits root)


## 7. Save and Reload

The trained tokenizer is saved as two plain-text files:
- `vocab.json` — token string to integer ID mapping
- `merges.txt` — learned BPE merge rules, one per line

In [17]:
save_dir = os.path.join(tmpdir, "my_tokenizer")
tok.save(save_dir)
print(f"Saved to {save_dir}/")
print(f"  vocab.json : {os.path.getsize(os.path.join(save_dir, 'vocab.json')):,} bytes")
print(f"  merges.txt : {os.path.getsize(os.path.join(save_dir, 'merges.txt')):,} bytes")

tok2 = TagalogTokenizer()
tok2.load(save_dir)

text = "Nagtatrabaho ang tatay sa opisina."
assert tok.encode(text) == tok2.encode(text)
print(f"\nReloaded tokenizer produces identical output: OK")

Saved to C:\Users\JOHNPA~1\AppData\Local\Temp\tmpnou52m43\my_tokenizer/
  vocab.json : 4,225 bytes
  merges.txt : 1,332 bytes

Reloaded tokenizer produces identical output: OK


## 8. Token Efficiency vs Character-Level

BPE should produce far fewer tokens than splitting every character individually.
The savings grow with word complexity.

In [18]:
sentences = [
    "Kumain siya ng pagkain sa hapagkainan.",
    "Maganda ang panahon ngayon kaya lumabas kami.",
    "Pinakamahusay ang ginawa niya sa pagtatanghal.",
    "Nakapagpapakain na ang nanay ng maraming bisita.",
]

print(f"  {'Sentence':<48} {'BPE':>5} {'Chars':>6} {'Savings':>8}")
print(f"  {'-'*48} {'-'*5} {'-'*6} {'-'*8}")
for sent in sentences:
    bpe_n = len(tok.encode(sent))
    char_n = len(sent.replace(" ", ""))
    savings = (char_n - bpe_n) / char_n * 100
    print(f"  {sent:<48} {bpe_n:>5} {char_n:>6} {savings:>7.0f}%")

  Sentence                                           BPE  Chars  Savings
  ------------------------------------------------ ----- ------ --------
  Kumain siya ng pagkain sa hapagkainan.              18     33      45%
  Maganda ang panahon ngayon kaya lumabas kami.       22     39      44%
  Pinakamahusay ang ginawa niya sa pagtatanghal.      18     41      56%
  Nakapagpapakain na ang nanay ng maraming bisita.    27     42      36%


## Next Steps

For a deeper comparison against GPT-2 and SentencePiece, see:
- `tokenizer_comparisons_fil.ipynb` — side-by-side on 20 Filipino sentences with charts
- `tokenizer_comparisons.ipynb` — full evaluation with the WikiText-TL-39 corpus